# Análisis de Correlaciones - Commodities y Predictores

**Objetivo:** Analizar relaciones entre commodities y predictores macroeconómicos

Este notebook:
- Carga datos procesados en formato wide
- Calcula matrices de correlación
- Identifica relaciones fuertes
- Visualiza heatmaps

In [ ]:
# Setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Importar módulos del proyecto
from src.data import process
from src.config import PROCESSED_DIR, FIGURES_DIR

# Configuración de visualización
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('coolwarm')
%matplotlib inline

pd.options.display.max_columns = 50

## 1. Cargar Datos Procesados

In [ ]:
# Verificar si existe el dataset procesado
processed_file = PROCESSED_DIR / 'commodities_base_daily.csv'

if processed_file.exists():
    print(f"✓ Cargando datos procesados desde {processed_file}")
    df = pd.read_csv(processed_file, parse_dates=['date'])
else:
    print("⚠ Dataset procesado no encontrado. Ejecutando pipeline de procesamiento...")
    df = process.main()

print(f"\n📊 Dimensiones: {df.shape}")
print(f"📅 Período: {df['date'].min().date()} → {df['date'].max().date()}")
df.head()

## 2. Seleccionar Columnas de Precios

Filtramos solo las columnas de precios originales (sin lags, rolling, returns)

In [ ]:
# Identificar columnas de precios base (sin features engineered)
exclude_patterns = ['lag', 'ma', 'std', 'return', 'year', 'month', 'quarter', 'day', 'week']

price_columns = [col for col in df.columns 
                 if col not in ['date'] and 
                 not any(pattern in col.lower() for pattern in exclude_patterns)]

print(f"📊 Columnas de precios identificadas: {len(price_columns)}")
print(f"\nColumnas: {price_columns[:10]}...")  # Mostrar primeras 10

# Crear subset solo con precios
df_prices = df[price_columns].copy()

## 3. Matriz de Correlación Completa

In [ ]:
# Calcular matriz de correlación
corr_matrix = df_prices.corr()

print(f"📊 Matriz de correlación: {corr_matrix.shape}")

# Heatmap completo
fig, ax = plt.subplots(figsize=(20, 18))

sns.heatmap(corr_matrix, 
            cmap='coolwarm', 
            center=0,
            vmin=-1, vmax=1,
            square=True,
            linewidths=0.5,
            cbar_kws={'shrink': 0.8, 'label': 'Correlación'},
            annot=False,
            fmt='.2f',
            ax=ax)

ax.set_title('Matriz de Correlación - Commodities y Predictores', fontsize=16, fontweight='bold', pad=20)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'correlacion_completa.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. Correlaciones Más Fuertes

Identificar pares con correlación |r| > 0.7

In [ ]:
# Extraer correlaciones fuertes
def get_strong_correlations(corr_matrix, threshold=0.7):
    """
    Encuentra pares con correlación absoluta mayor al threshold
    """
    # Crear máscara para triángulo superior (evitar duplicados)
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
    
    # Extraer correlaciones
    strong_corr = []
    for i in range(len(corr_matrix)):
        for j in range(i+1, len(corr_matrix)):
            if abs(corr_matrix.iloc[i, j]) >= threshold:
                strong_corr.append({
                    'Variable 1': corr_matrix.index[i],
                    'Variable 2': corr_matrix.columns[j],
                    'Correlación': corr_matrix.iloc[i, j]
                })
    
    return pd.DataFrame(strong_corr).sort_values('Correlación', key=abs, ascending=False)

strong_corr_df = get_strong_correlations(corr_matrix, threshold=0.7)

print(f"\n🔍 Pares con correlación |r| >= 0.7: {len(strong_corr_df)}\n")
strong_corr_df.head(20)

## 5. Correlaciones con Predictores

Analizar específicamente cómo cada predictor se correlaciona con commodities

In [ ]:
# Identificar predictores (asumiendo que tienen nombres conocidos)
predictor_names = ['VIX', 'DXY', 'SP500', 'TNX', 'TLT']  # Ajustar según predictores reales

# Filtrar predictores que existan en el dataset
available_predictors = [p for p in predictor_names if p in df_prices.columns]

if available_predictors:
    # Columnas de commodities (excluir predictores)
    commodity_cols = [col for col in price_columns if col not in available_predictors]
    
    print(f"📊 Predictores disponibles: {available_predictors}")
    print(f"📊 Commodities: {len(commodity_cols)}\n")
    
    # Correlaciones de cada predictor con commodities
    for predictor in available_predictors:
        print(f"\n{'='*60}")
        print(f"Predictor: {predictor}")
        print(f"{'='*60}")
        
        pred_corr = corr_matrix.loc[commodity_cols, predictor].sort_values(key=abs, ascending=False)
        
        print(f"\nTop 10 correlaciones más fuertes con {predictor}:")
        print(pred_corr.head(10))
        
        # Visualización
        fig, ax = plt.subplots(figsize=(10, max(8, len(commodity_cols)*0.3)))
        
        colors = ['darkred' if x < 0 else 'darkgreen' for x in pred_corr.values]
        pred_corr.plot(kind='barh', ax=ax, color=colors, edgecolor='black', linewidth=0.5)
        
        ax.set_title(f'Correlación de {predictor} con Commodities', fontsize=14, fontweight='bold')
        ax.set_xlabel('Correlación de Pearson')
        ax.axvline(x=0, color='black', linewidth=1)
        ax.grid(alpha=0.3, axis='x')
        
        plt.tight_layout()
        plt.savefig(FIGURES_DIR / f'correlacion_{predictor.lower()}_commodities.png', dpi=300, bbox_inches='tight')
        plt.show()
else:
    print("⚠ No se encontraron predictores en el dataset")

## 6. Clustermap - Agrupación Jerárquica

In [ ]:
# Clustermap para identificar grupos de commodities con comportamiento similar
g = sns.clustermap(corr_matrix, 
                   cmap='coolwarm', 
                   center=0,
                   vmin=-1, vmax=1,
                   figsize=(18, 18),
                   linewidths=0.5,
                   cbar_kws={'label': 'Correlación'},
                   annot=False,
                   fmt='.2f')

g.fig.suptitle('Clustermap - Agrupación Jerárquica de Correlaciones', 
               fontsize=16, fontweight='bold', y=0.98)

plt.savefig(FIGURES_DIR / 'correlacion_clustermap.png', dpi=300, bbox_inches='tight')
plt.show()

## 7. Exportar Resultados

In [ ]:
# Guardar matriz de correlación
corr_matrix.to_csv(PROCESSED_DIR / 'correlation_matrix.csv')
print(f"✓ Matriz de correlación guardada: {PROCESSED_DIR / 'correlation_matrix.csv'}")

# Guardar correlaciones fuertes
strong_corr_df.to_csv(PROCESSED_DIR / 'strong_correlations.csv', index=False)
print(f"✓ Correlaciones fuertes guardadas: {PROCESSED_DIR / 'strong_correlations.csv'}")

## 8. Conclusiones

**Insights clave del análisis de correlaciones:**

1. **Commodities altamente correlacionados:** Identificar grupos (ej: metales preciosos, energía)
2. **Relación con predictores:** Evaluar poder predictivo de VIX, DXY, etc.
3. **Diversificación:** Commodities con baja correlación son buenos para portfolios
4. **Clusters:** Grupos jerárquicos revelan patrones de comportamiento conjunto

**Próximos análisis:**
- Correlaciones móviles en el tiempo
- Análisis de cointegración
- Modelos de predicción multivariados